In [ ]:
import json
import time

import requests
from requests.auth import HTTPBasicAuth

from eodag import EODataAccessGateway


In [ ]:
GRASS_PROJECT = "s2_epsg32635"
START_YEAR = 2025
START_MONTH = 8
END_YEAR = 2025
END_MONTH = 8
TILE_ID = "35SKC"


In [2]:
# variables to set the actinia host, version, and user

actinia_baseurl = "http://localhost:8088"
actinia_version = "v3"
actinia_url = actinia_baseurl + "/api/" + actinia_version
actinia_auth = HTTPBasicAuth('actinia', 'actinia')  # username, password


In [57]:
search_criteria = {
    "productType": "S2MSI2A",
    "start": f"{START_YEAR}-{START_MONTH:02d}-01",
    "end": f"{END_YEAR}-{END_MONTH:02d}-05",
    "tileIdentifier": TILE_ID,
}


In [58]:
dag = EODataAccessGateway()
all_products = dag.search_all(**search_criteria)
s2_ids = [all_products[i].properties['id']  for i in range(len(all_products))]
print(s2_ids)

['S2C_MSIL2A_20250802T090611_N0511_R050_T35SKC_20250802T124515', 'S2A_MSIL2A_20250804T091331_N0511_R050_T35SKC_20250804T120007']


In [9]:
# helper function to print formatted JSON using the json module

def print_as_json(data):
    print(json.dumps(data, indent=2))

# helper function to verify a request
def verify_request(request, success_code=200):
    if request.status_code != success_code:
        print("ERROR: actinia processing failed with status code %d!" % request.status_code)
        print("See errors below:")
        print_as_json(request.json())
        request_url = request.json()["urls"]["status"]
        requests.delete(url=request_url, auth=actinia_auth)
        raise Exception("The resource <%s> has been terminated." % request_url)

In [10]:


# make a GET request to the actinia data API
request_url = actinia_url + "/locations"
print("actinia GET request:")
print(request_url)
print("---")
request = requests.get(url=request_url, auth=actinia_auth)

# check if anything went wrong
verify_request(request, 200)

# get a json-encoded content of the response
jsonResponse = request.json()

print("Available locations:")

# print formatted JSON
print_as_json(jsonResponse)



actinia GET request:
http://localhost:8088/api/v3/locations
---
Available locations:
{
  "projects": [
    "test_epsg32632",
    "test_loc_epsg4326",
    "S2_processing_epsg32632",
    "testtest4326",
    "testtest_4326",
    "test4326"
  ],
  "status": "success"
}


In [54]:
# make a POST request to the actinia data API
request_url = f"{actinia_url}/locations/{GRASS_PROJECT}/mapsets/PERMANENT/processing"
print("actinia POST request:")
print(request_url)
print("---")

actinia POST request:
http://localhost:8088/api/v3/locations/s2_epsg32635/mapsets/PERMANENT/processing
---


In [56]:
for s2_id in s2_ids:
    print(f"Processing Sentinel-2 scene ID: {s2_id}")
    process_chain = json.load(open("pc_MAIN.json"))
    process_chain["list"][0]["inputs"][0]["value"] = s2_id
    process_chain["list"][3]["inputs"][2]['value'] = f"NDVI_{s2_id.split('_')[2][:8]}_{s2_id.split('_')[0]}"
    # print("Using process chain:")
    # print_as_json(process_chain)

    request = requests.post(url=request_url, auth=actinia_auth, json=process_chain)
    # check if anything went wrong
    verify_request(request, 200)


    # get a json-encoded content of the response
    jsonResponse = request.json()

    print(jsonResponse)

    while jsonResponse["status"] in ("accepted", "running"):
        status_request_url = jsonResponse["urls"]["status"]
        status_request = requests.get(url=status_request_url.replace("https", "http"), auth=actinia_auth)
        jsonResponse = status_request.json()
        print(f"Polling status for {s2_id}: {jsonResponse['status']}")
        print(f"Doing: {jsonResponse['message']}")
        print('#########################################')
        time.sleep(30)

    print(f"Final status for {s2_id}: {jsonResponse['status']}")




Processing Sentinel-2 scene ID: S2C_MSIL2A_20250802T090611_N0511_R050_T35SKC_20250802T124515
{'accept_datetime': '2025-12-10 12:11:26.816081', 'accept_timestamp': 1765368686.816079, 'api_info': {'endpoint': 'gdiasyncpersistentresource', 'method': 'POST', 'path': '/api/v3/locations/s2_epsg32635/mapsets/PERMANENT/processing', 'request_url': 'http://localhost:8088/api/v3/locations/s2_epsg32635/mapsets/PERMANENT/processing'}, 'datetime': '2025-12-10 12:11:27.516540', 'http_code': 200, 'message': 'Resource accepted', 'process_chain_list': [], 'process_results': {}, 'queue': 'local', 'resource_id': 'resource_id-44c3feff-939c-4f44-84a3-7ba6f834e254', 'status': 'accepted', 'time_delta': 0.7004988193511963, 'timestamp': 1765368687.5165281, 'urls': {'resources': [], 'status': 'https://localhost:8088/api/v3/resources/actinia/resource_id-44c3feff-939c-4f44-84a3-7ba6f834e254'}, 'user_id': 'actinia'}
Polling status for S2C_MSIL2A_20250802T090611_N0511_R050_T35SKC_20250802T124515: accepted
Doing: Res